In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
!pip install tqdm

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torchinfo import summary

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
# load test,train data
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
#processdata
drop = ["Name","Sex","Embarked","Cabin","Ticket"]
# catagorize = ["Sex","Embarked","Cabin"]
catagorize = []

def process_and_filter(df,category = None):
    category_set = []
    df = df.drop(columns=[c for c in drop if c in df.columns]).copy()
    # for i in catagorize:
        # df[i] = df[i].astype("category").cat.codes +1
    # df[catagorize] = df[catagorize].astype("category").cat.codes+1
    if category:
        for col in category:
            test[col] = pd.Categorical(test[col], categories=categories).codes + 1
            # df[cols] = df[cols].apply(lambda col: col.astype("category").cat.codes + 1)
    else:
        for cols in catagorize:
            df[cols] = df[cols].apply(lambda col: col.astype("category").cat.codes + 1)
    df = df.fillna(df.mean(numeric_only=True))

    return df , category_set

train , category_set = process_and_filter(train)
# categories = train.cat.categories
test ,catagory_set = process_and_filter(test)
# test[col] = pd.Categorical(test[col], categories=categories).codes + 1
train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
0,1,0,3,22.0,1,0,7.2500
1,2,1,1,38.0,1,0,71.2833
2,3,1,3,26.0,0,0,7.9250
3,4,1,1,35.0,1,0,53.1000
4,5,0,3,35.0,0,0,8.0500


In [5]:
test.head()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
0,892,3,34.5,0,0,7.8292
1,893,3,47.0,1,0,7.0000
2,894,2,62.0,0,0,9.6875
3,895,3,27.0,0,0,8.6625
4,896,3,22.0,1,1,12.2875


In [6]:
class DataFrameDataset(Dataset):
    def __init__(self,df,target:str ):
        try:
            self.features = torch.tensor(df.drop(columns=[target]).values,dtype = torch.float32)
            self.labels = torch.tensor(df[target].values,dtype=torch.long)
        except:
            self.features = torch.tensor(df.values,dtype = torch.float32)
            
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self,idx):
        try:
            return self.features[idx], self.labels[idx]
        except:
            return self.features[idx]
train_torch_dataset = DataFrameDataset(train,"Survived")
dataloader = DataLoader(train_torch_dataset,batch_size=32,shuffle=True)
test_torch_dataset = DataFrameDataset(test,None)
test_dataloader  = DataLoader(test_torch_dataset,batch_size=32,shuffle=False)

print(dataloader)

In [7]:

class NeuralNet(nn.Module):
    def __init__(self,in_dim):
        super().__init__()
        self.runthrough = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            # nn.Sigmoid()
        )
    def forward(self,x):
        ret = self.runthrough(x)
        return ret

in_dim_df = len(train.columns) - 1
model = NeuralNet(in_dim = in_dim_df)
summary(model)
# print(in_dim_df)

Layer (type:depth-idx)                   Param #
NeuralNet                                --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       448
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       130
Total params: 578
Trainable params: 578
Non-trainable params: 0

In [8]:
#hyperparameters
import torch.optim as optim 
from tqdm import tqdm
epoch = 1000
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
loss_function = nn.CrossEntropyLoss()
train_loss = 0
pbar = tqdm(range(epoch), desc="Training")
for i in pbar:
    model.train()
    train_loss = 0
    for x , y in dataloader:
        
        optimizer.zero_grad()
        predictions = model(x)
        # print(predictions.dtype,y.dtype)
        loss = loss_function(predictions,y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    pbar.set_postfix(train_loss=f"{train_loss:.4f}")

        

Training: 100%|██████████| 1000/1000 [00:24<00:00, 41.07it/s, train_loss=15.7718]


In [9]:
model.eval()
predictions_total = []
sigmoid = nn.Sigmoid()  # Capital 'S' for the layer class

with torch.no_grad():
  # Note: If test_dataloader doesn't have labels (y), use `for x in test_dataloader:`
  for x in test_dataloader:
    logits = model(x)
    preds = torch.argmax(logits, dim=1)

    # Move to CPU, convert to numpy, and flatten to a 1D array
    predictions_total.extend(preds.cpu().numpy().flatten())

# Ensure lengths match before creating the DataFrame
print(len(predictions_total),len(test["PassengerId"]))
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions_total,
})

output.to_csv('submission.csv', index=False)

418 418


In [10]:
sub = pd.read_csv('submission.csv')
print(sub.shape)  # Must be (418, 2)
print(sub.head())  # Columns must be: PassengerId, Survived
print(sub['Survived'].value_counts(dropna=False))  # Only 0s and 1s, no NaNs


(418, 2)
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
Survived
1    284
0    134
Name: count, dtype: int64
